In [ ]:
from libs import *
from consts import *
from basic_funcs import *
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from astropy.visualization import ZScaleInterval, ImageNormalize
vosclient = Client()

def view_maskdata(tilename: str, grid=False):
    """
    Plots the full mask where:
    Bright = good pixels (unmasked, usable)
    Dark = bad pixels (masked, unusable)
    with a 10x10 grid overlay.
    """
    mask = get_mask(tilename)
    ny, nx = mask.shape  # image dimensions

    plt.figure(figsize=(6, 6))
    plt.imshow(mask, cmap='gray', origin='lower')  # white=true, black=false
    plt.title(f"Mask Data for {tilename} (Bright = Good, Dark = Masked)")
    
    if grid:
        n_grid = 10
        x_ticks = np.linspace(0, nx, n_grid + 1)
        y_ticks = np.linspace(0, ny, n_grid + 1)
    
        for x in x_ticks:
            plt.axvline(x=x, color='red', linewidth=0.8, linestyle='--', alpha=0.6)
        for y in y_ticks:
            plt.axhline(y=y, color='red', linewidth=0.8, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

def view_fits(tilename: str, grid=False):
    #get_fits(tilename)
    source = f"vos:cfis/tiles_LSB_DR5/{tilename}.fits"

    arc_dir = "vos://cadc.nrc.ca~arc/home/snyr617/fits"
    local_cache = os.path.join(os.getcwd(), "fits")
    os.makedirs(local_cache, exist_ok=True)

    fits_file = os.path.join(local_cache, f"{tilename}.fits")
    if not os.path.exists(fits_file):
        print("Downloading")
        vosclient.copy(source, fits_file)
        print("Download complete")
    else:
        print("Already in folder")
    #fits_file = Path(FITS_DIR) / f"{tilename}.fits"

    hdul = fits.open(fits_file)
    data = hdul[0].data

    norm = ImageNormalize(data, interval=ZScaleInterval())
    plt.figure(figsize=(6, 6))
    plt.imshow(data, cmap='gray', origin='lower', norm=norm)
    plt.title(tilename)

    if grid:
        n_grid = 10
        x_ticks = np.linspace(0, 10000, n_grid + 1)
        y_ticks = np.linspace(0, 10000, n_grid + 1)
        for x in x_ticks:
            plt.axvline(x=x, color='red', linewidth=0.8, linestyle='--', alpha=0.6)
        for y in y_ticks:
            plt.axhline(y=y, color='red', linewidth=0.8, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.show()

    os.remove(fits_file)

def view_catalogue(tilename: str, grid=False, cut=False):
    if not cut:
        tile_file = Path(LSB_CAT_DIR) / f"{tilename}.cat"
        tilepath = os.path.join(LSB_CAT_DIR, tilename + ".cat")

        if not tile_file.exists():
            get_lsb_cat(tilename)
        
        df = read_cat(tilepath)
    else:
        gridded_path = Path(GRID_FLAG_DIR) / (tilename + GRID_FLAG_END)
        gridded_df = pd.read_csv(gridded_path)

        df = gridded_df[
            (gridded_df["GOOD_REGION"]) &
            (gridded_df[MAG_ABOVE]) &
            (gridded_df[MAG_BELOW]) &
            (gridded_df[SIZE_ABOVE])
        ]
    
    plt.figure(figsize=(6, 6))
    plt.scatter(df["X_IMAGE"], df["Y_IMAGE"], s=3)
    plt.xlabel("X_IMAGE (pixels)")
    plt.ylabel("Y_IMAGE (pixels)")
    plt.title("Catalogue Objects")
    plt.xlim(0, 10000)
    plt.ylim(0, 10000)
    if grid:
        n_grid = 10
        x_ticks = np.linspace(0, 10000, n_grid + 1)
        y_ticks = np.linspace(0, 10000, n_grid + 1)
        for x in x_ticks:
            plt.axvline(x=x, color='red', linewidth=0.8, linestyle='--', alpha=0.6)
        for y in y_ticks:
            plt.axhline(y=y, color='red', linewidth=0.8, linestyle='--', alpha=0.6)
    plt.tight_layout()
    plt.show()

def view_all(tilename: str):
    view_maskdata(tilename)
    view_fits(tilename)
    view_catalogue(tilename)

def view_masking(tilename: str, grid=False, cut=False):
    mask = get_mask(tilename)
    ny, nx = mask.shape  # image dimensions
    
    if not cut:
        tilepath = os.path.join(LSB_CAT_DIR, tilename + ".cat")
        df = read_cat(tilepath)
    else:
        gridded_path = Path(GRID_FLAG_DIR) / (tilename + GRID_FLAG_END)
        gridded_df = pd.read_csv(gridded_path)

        df = gridded_df[
            (gridded_df["GOOD_REGION"]) &
            (gridded_df[MAG_ABOVE]) &
            (gridded_df[MAG_BELOW]) &
            (gridded_df[SIZE_ABOVE])
        ]
    
    plt.figure(figsize=(6, 6))
    plt.scatter(df["X_IMAGE"], df["Y_IMAGE"], s=2, zorder=1)

    cm = mcolors.ListedColormap([
        (0, 0, 0, 1.0),   # false: black, opaque
        (0, 0, 0, 0.0)    # true: fully transparent
    ])
    plt.imshow(mask, cmap=cm, origin='lower', zorder=2)
    plt.xlabel("X_IMAGE (pixels)")
    plt.ylabel("Y_IMAGE (pixels)")
    plt.title(f"Mask and Catalogue Data for {tilename}")
    plt.xlim(0, nx)
    plt.ylim(0, ny)
    if grid:
        n_grid = 10
        x_ticks = np.linspace(0, nx, n_grid + 1)
        y_ticks = np.linspace(0, ny, n_grid + 1)
    
        for x in x_ticks:
            plt.axvline(x=x, color='red', linewidth=0.8, linestyle='--', alpha=0.6, zorder=3)
        for y in y_ticks:
            plt.axhline(y=y, color='red', linewidth=0.8, linestyle='--', alpha=0.6, zorder=3)

    plt.tight_layout()
    plt.show()

def view_combined(tilename: str, grid=False, cut=False):
    source = f"vos:cfis/tiles_LSB_DR5/{tilename}.fits"

    arc_dir = "vos://cadc.nrc.ca~arc/home/snyr617/fits"
    local_cache = os.path.join(os.getcwd(), "fits")
    os.makedirs(local_cache, exist_ok=True)

    local_file = os.path.join(local_cache, f"{tilename}.fits")
    if not os.path.exists(local_file):
        print("Downloading")
        vosclient.copy(source, local_file)
        print("Download complete")
    else:
        print("Already in folder")

    hdul = fits.open(local_file)
    data = hdul[0].data

    mask = get_mask(tilename)
    ny, nx = mask.shape

    if not cut:
        tilepath = os.path.join(LSB_CAT_DIR, tilename + ".cat")
        df = read_cat(tilepath)
    else:
        gridded_path = Path(GRID_FLAG_DIR) / (tilename + GRID_FLAG_END + ".csv")
        gridded_df = pd.read_csv(gridded_path)
        df = gridded_df[
            (gridded_df["GOOD_REGION"]) &
            (gridded_df["MAG_ABOVE_22"]) &
            (gridded_df["MAG_BELOW_25"]) &
            (gridded_df["SIZE_ABOVE_2.7"])
        ]

    plt.figure(figsize=(7, 7))

    norm = ImageNormalize(data, interval=ZScaleInterval())
    plt.imshow(data, cmap='gray', origin='lower', norm=norm, zorder=0)

    plt.scatter(df["X_IMAGE"], df["Y_IMAGE"], s=3, zorder=2)

    cm = mcolors.ListedColormap([
        (0, 0, 0, 1.0),  # 0 = black opaque
        (0, 0, 0, 0.0)   # 1 = transparent
    ])
    plt.imshow(mask, cmap=cm, origin='lower', zorder=1)

    # Set title and limits
    plt.title(f"FITS + Mask + Catalogue for {tilename}")
    plt.xlim(0, nx)
    plt.ylim(0, ny)

    if grid:
        n_grid = 10
        x_ticks = np.linspace(0, nx, n_grid + 1)
        y_ticks = np.linspace(0, ny, n_grid + 1)
        for x in x_ticks:
            plt.axvline(x=x, color='red', linewidth=0.8, linestyle='--', alpha=0.5, zorder=3)
        for y in y_ticks:
            plt.axhline(y=y, color='red', linewidth=0.8, linestyle='--', alpha=0.5, zorder=3)

    plt.tight_layout()
    plt.show()